# Lilium Tower

## Parametrization and Channel Generation

In [ ]:
import MeshFEM, mesh, sparse_matrices, benchmark, field_sampler, mesh_utilities
import inflatables_parametrization as parametrization, numpy as np, importlib, pickle, wall_generation
import utils
import py_newton_optimizer
from py_newton_optimizer import NewtonOptimizerOptions
from numpy.linalg import norm
from io_redirection import suppress_stdout
import visualization, wall_width_formulas as wwf

target_surf = mesh.Mesh('../examples/lilium.msh')
target_surf.setVertices(utils.prototypeScaleNormalization(target_surf.vertices(), placeAtopFloor=True))
target_surf = mesh_utilities.subdivide_loop(target_surf, 1)

In [ ]:
# Choose reasonable stretching bounds
alphaMin = wwf.stretchFactorForCanonicalWallWidth(wwf.canonicalWallWidthForGeometry(2, 10))
alphaMax = wwf.stretchFactorForCanonicalWallWidth(wwf.canonicalWallWidthForGeometry(1, 10))
print(alphaMin, alphaMax)

In [ ]:
lg = parametrization.LocalGlobalParametrizer(target_surf, parametrization.lscm(target_surf))

for i in range(1000): lg.runIteration()
print(lg.energy())
lg.alphaMin = 1.4
lg.alphaMax = np.pi / 2

print(lg.energy())
lg.runIteration()
print(lg.energy())

In [ ]:
rparam = parametrization.RegularizedParametrizerSVD(lg)
for i in range(5000): lg.runIteration()
print(lg.energy())

In [ ]:
rparam = parametrization.RegularizedParametrizerSVD(target_surf, lg.uv())
rparam.alphaMin = alphaMin
rparam.alphaMax = alphaMax

In [ ]:
opts = NewtonOptimizerOptions()

In [ ]:
def optimize_rparam(param, alphaRegW, phiRegW, bendRegW):
    param.alphaRegW = alphaRegW
    param.phiRegW = phiRegW
    param.bendRegW = bendRegW
    opts = NewtonOptimizerOptions()
    opts.niter = 2000
    opts.hessianProjectionController = py_newton_optimizer.HessianProjectionAdaptive()
    #opts.hessianProjectionController = py_newton_optimizer.HessianProjectionNever()
    cr = parametrization.regularized_parametrization_newton(param, param.rigidMotionPinVars, opts)

In [ ]:
benchmark.reset()
with suppress_stdout(): optimize_rparam(rparam, 1e-2, 1e-2, 0)
with suppress_stdout(): optimize_rparam(rparam, 1e-3, 1e-3, 0)
with suppress_stdout(): optimize_rparam(rparam, 1e-4, 1e-4, 0)
with suppress_stdout(): optimize_rparam(rparam, 1e-5, 1e-5, 0)
benchmark.report()

In [ ]:
PET = parametrization.RegularizedParametrizerSVD.EnergyType
list(map(rparam.energy, [PET.Fitting, PET.AlphaRegularization, PET.PhiRegularization]))

In [ ]:
visualization.visualize(rparam)

In [ ]:
importlib.reload(visualization)
visualization.visualizeChannelOrientation(rparam, quiver=visualization.QuiverVisualization.PER_VTX, orientationHue=False)

In [ ]:
visualization.singularValueHistogram(rparam)

In [ ]:
widths = wwf.wallWidthForCanonicalWidth(wwf.canonicalWallWidthForStretchFactor(rparam.getAlphas()), 10)
(np.min(widths), np.max(widths))

## Upsampling and channel generation

In [ ]:
nsubdiv=4
upsampledMesh, upsampledAngles, upsampledStretches = rparam.upsampledVertexLeftStretchAnglesAndMagnitudes(nsubdiv)
(sdfVertices, sdfTris, sdf) = wall_generation.evaluate_stripe_field(upsampledMesh.vertices(), upsampledMesh.triangles(), upsampledAngles,
                                                                    wwf.canonicalWallWidthForStretchFactor(upsampledStretches), frequency=100)
#pickle.dump((sdfVertices, sdfTris, sdf), open('stripe_sdf_ns4_f100.pkl', 'wb'))

In [ ]:
import pickle, mesh, wall_generation, visualization, numpy as np
#(sdfVertices, sdfTris, sdf) = pickle.load(open('stripe_sdf_ns4_f100.pkl', 'rb'))

In [ ]:
visualization.scalarFieldPlotFast(sdfVertices, sdfTris, sdf, height=12)

In [ ]:
pts, edges = wall_generation.extract_contours(sdfVertices, sdfTris, sdf,
                                              targetEdgeSpacing=0.01,
                                              minContourLen=0.075)

In [ ]:
visualization.plot_line_segments(pts, edges, width=20, height=16)

## Meshing and inflation simulation

In [ ]:
m, fuseMarkers, edgeMarkers = wall_generation.triangulate_channel_walls(pts[:,0:2], edges, 0.0002)
visualization.plot_2d_mesh(m, pointList=np.where(np.array(fuseMarkers) == 1)[0], width=20, height=18)

In [ ]:
import inflation
isheet = inflation.InflatableSheet(m, np.array(fuseMarkers) != 0)

In [ ]:
paramSampler = field_sampler.FieldSampler(np.pad(rparam.uv(), [(0, 0), (0, 1)], 'constant'), lilium.triangles())
liftedSheetPositions = paramSampler.sample(m.vertices(), lilium.vertices())

In [ ]:
isheet.setUninflatedDeformation(liftedSheetPositions.transpose())

In [ ]:
import py_newton_optimizer
niter = 2000
iterations_per_output = 10
opts = py_newton_optimizer.NewtonOptimizerOptions()
opts.useIdentityMetric = True
opts.beta = 1e-4
opts.gradTol = 1e-10
opts.niter = iterations_per_output

In [ ]:
# Are the flat region causing a problem? They might not actually control the metric...
# Try replacing them with single wall...
# Analyze the actual stretching factor (much easier to do with skeleton walls)

In [ ]:
from tri_mesh_viewer import TriMeshViewer
viewer = TriMeshViewer(isheet.visualizationMesh(), width=768, height=640)
viewer.showWireframe()
viewer.show()

In [ ]:
isheet.setUseTensionFieldEnergy(True)

In [ ]:
import time
isheet.pressure = 35
benchmark.reset()
for step in range(int(niter / iterations_per_output)):
    cr = inflation.inflation_newton(isheet, isheet.rigidMotionPinVars, opts)
    if cr.numIters() < iterations_per_output: break
    viewer.update(False, isheet.visualizationMesh())
    time.sleep(0.05) # Allow some mesh synchronization time for pythreejs
benchmark.report()

In [ ]:
isheet.tensionStateHistogram()

In [ ]:
benchmark.report()